# NHS Referral-to-Treatment Waiting Times Analysis
## Data Cleaning

**Dataset:** NHS England Consultant-led Referral to Treatment (RTT), July 2026 — Incomplete Provider data.

### Notebook purpose

This notebook converts the raw NHS England workbook into a clean, analysis-ready dataset for SQL, Python analysis and Power BI.

The cleaning workflow will:

1. reload the raw `Provider` worksheet using the verified header row;
2. remove only fully empty columns;
3. standardise column names;
4. rename all 105 weekly waiting-time bands;
5. convert `-` values in waiting-time statistics to missing values;
6. convert percentage fields from decimal proportions to percentage points;
7. create useful derived KPIs;
8. add reporting-period metadata;
9. label provider-total and treatment-function rows;
10. validate the cleaned data against the original NHS measures; and
11. export a processed CSV.

> The raw workbook is never overwritten.


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)


## Define file paths

The project keeps source files and processed files separate so the original NHS extract remains unchanged.


In [2]:
raw_file = Path("../data/raw/Incomplete-Provider-Jul26-XLSX-9M-97ylx5.xlsx")
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_file = processed_dir / "rtt_incomplete_provider_jul_2026_clean.csv"


## Load the verified raw worksheet

From `01_data_understanding.ipynb`, we established that:

- `Provider` is the correct starting worksheet;
- the real header begins on Excel row 14, so pandas uses `header=13`; and
- the worksheet contains one completely blank column that can safely be removed.


In [3]:
df = pd.read_excel(
    raw_file,
    sheet_name="Provider",
    header=13
)

df = df.dropna(axis=1, how="all")

print("Shape:", df.shape)
df.head()


Shape: (3504, 119)


,Region Code,Provider Code,Provider Name,Treatment Function Code,Treatment Function,>0-1,>1-2,>2-3,>3-4,>4-5,>5-6,>6-7,>7-8,>8-9,>9-10,>10-11,>11-12,>12-13,>13-14,>14-15,>15-16,>16-17,>17-18,>18-19,>19-20,>20-21,>21-22,>22-23,>23-24,>24-25,>25-26,>26-27,>27-28,>28-29,>29-30,>30-31,>31-32,>32-33,>33-34,>34-35,>35-36,>36-37,>37-38,>38-39,>39-40,>40-41,>41-42,>42-43,>43-44,>44-45,>45-46,>46-47,>47-48,>48-49,>49-50,>50-51,>51-52,>52-53,>53-54,>54-55,>55-56,>56-57,>57-58,>58-59,>59-60,>60-61,>61-62,>62-63,>63-64,>64-65,>65-66,>66-67,>67-68,>68-69,>69-70,>70-71,>71-72,>72-73,>73-74,>74-75,>75-76,>76-77,>77-78,>78-79,>79-80,>80-81,>81-82,>82-83,>83-84,>84-85,>85-86,>86-87,>87-88,>88-89,>89-90,>90-91,>91-92,>92-93,>93-94,>94-95,>95-96,>96-97,>97-98,>98-99,>99-100,>100-101,>101-102,>102-103,>103-104,104 plus,Total number of incomplete pathways,Total within 18 weeks,% within 18 weeks,Average (median) waiting time (in weeks),92nd percentile waiting time (in weeks),Total 52 plus weeks,Total 78 plus weeks,Total 65 plus weeks,% 52 plus weeks
0,Y56,R1H,BARTS HEALTH NHS TRUST,C_100,General Surgery Service,257,238,194,211,193,198,193,149,169,105,153,113,131,133,128,122,81,147,105,95,104,71,71,62,75,54,55,50,60,51,26,36,41,40,35,48,32,23,49,45,42,31,46,30,27,31,20,19,15,17,4,11,6,7,2,2,5,4,10,5,1,1,1,1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4483,2915,0.650234,12.522901,39.585778,47,0,2,0.010484
1,Y56,R1H,BARTS HEALTH NHS TRUST,C_101,Urology Service,359,239,191,187,207,226,150,296,183,132,220,192,163,185,204,123,113,97,153,121,146,106,126,101,129,108,110,102,114,70,36,69,92,111,130,63,48,52,55,40,26,50,37,38,17,21,22,45,14,23,11,10,4,4,7,6,2,1,1,1,3,2,2,2,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5901,3467,0.587528,14.10049,37.517692,38,0,3,0.006440
2,Y56,R1H,BARTS HEALTH NHS TRUST,C_110,Trauma and Orthopaedic Service,396,321,329,298,300,248,272,349,301,210,320,259,225,253,260,213,205,252,204,164,170,160,139,137,159,148,176,175,190,151,52,80,130,146,138,117,101,101,87,89,95,93,76,63,45,51,50,33,24,24,16,9,9,9,8,11,13,4,6,5,3,2,1,0,2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8678,5011,0.577437,14.992308,39.536629,74,0,1,0.008527
3,Y56,R1H,BARTS HEALTH NHS TRUST,C_120,Ear Nose and Throat Service,253,229,188,202,232,191,216,182,160,119,145,149,117,165,131,138,66,119,109,90,110,109,113,103,126,109,127,137,125,109,34,58,113,111,99,81,84,84,78,69,90,88,63,66,91,58,57,53,40,56,36,40,32,31,20,23,27,22,19,19,13,11,11,14,4,2,3,4,3,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6180,3002,0.485761,18.807339,46.870175,262,0,16,0.042395
4,Y56,R1H,BARTS HEALTH NHS TRUST,C_130,Ophthalmology Service,335,277,314,329,240,292,264,262,234,180,255,241,171,225,218,205,111,158,132,130,125,106,88,115,96,100,78,101,75,73,40,67,63,62,50,70,59,44,38,54,56,30,21,21,9,11,3,7,3,6,1,0,0,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6249,4311,0.689870,11.591286,33.791613,4,0,0,0.000640


Expected shape after removing the fully blank column:

**3,504 rows × 119 columns**


## Preserve a copy of the loaded raw table

This copy is useful for before/after validation during cleaning.


In [4]:
raw_df = df.copy()


## Identify the 105 weekly waiting-time columns

The weekly bands range from `>0-1` to `104 plus`. Rather than manually listing 105 columns, identify them programmatically.


In [5]:
weekly_raw = [
    col for col in df.columns
    if re.match(r"^>\d+-\d+$", str(col)) or col == "104 plus"
]

print("Weekly band count:", len(weekly_raw))
print("First five:", weekly_raw[:5])
print("Last five:", weekly_raw[-5:])


Weekly band count: 105
First five: ['>0-1', '>1-2', '>2-3', '>3-4', '>4-5']
Last five: ['>100-101', '>101-102', '>102-103', '>103-104', '104 plus']


## Standardise column names

Clear, consistent `snake_case` names make the dataset easier to query in SQL and work with in Python and Power BI.

Examples:

- `Provider Code` → `provider_code`
- `Total number of incomplete pathways` → `total_waiting`
- `>17-18` → `wait_17_18_weeks`
- `104 plus` → `wait_104_plus_weeks`


In [6]:
rename_map = {
    "Region Code": "region_code",
    "Provider Code": "provider_code",
    "Provider Name": "provider_name",
    "Treatment Function Code": "treatment_function_code",
    "Treatment Function": "treatment_function",
    "Total number of incomplete pathways": "total_waiting",
    "Total within 18 weeks": "within_18_weeks",
    "% within 18 weeks": "pct_within_18_weeks",
    "Average (median) waiting time (in weeks)": "median_wait_weeks",
    "92nd percentile waiting time (in weeks)": "p92_wait_weeks",
    "Total 52 plus weeks": "over_52_weeks",
    "Total 65 plus weeks": "over_65_weeks",
    "Total 78 plus weeks": "over_78_weeks",
    "% 52 plus weeks": "pct_over_52_weeks",
}

for col in weekly_raw:
    if col == "104 plus":
        rename_map[col] = "wait_104_plus_weeks"
    else:
        start, end = re.findall(r"\d+", col)
        rename_map[col] = f"wait_{start}_{end}_weeks"

df = df.rename(columns=rename_map)

df.columns[:15].tolist()


['region_code',
 'provider_code',
 'provider_name',
 'treatment_function_code',
 'treatment_function',
 'wait_0_1_weeks',
 'wait_1_2_weeks',
 'wait_2_3_weeks',
 'wait_3_4_weeks',
 'wait_4_5_weeks',
 'wait_5_6_weeks',
 'wait_6_7_weeks',
 'wait_7_8_weeks',
 'wait_8_9_weeks',
 'wait_9_10_weeks']

## Handle `-` values in waiting-time statistics

The median and 92nd-percentile fields contain `-` in records where a numeric statistic is not supplied.

These values should not be treated as zero because zero would be a real waiting-time value. Instead, convert `-` to `NaN` and then convert the columns to numeric.


In [7]:
numeric_wait_columns = [
    "median_wait_weeks",
    "p92_wait_weeks"
]

for col in numeric_wait_columns:
    text_values = df[col].astype("string").str.strip()
    df[col] = pd.to_numeric(
        text_values.mask(text_values.eq("-")),
        errors="coerce"
    )

df[numeric_wait_columns].dtypes


median_wait_weeks    Float64
p92_wait_weeks       Float64
dtype: object

In [8]:
df[numeric_wait_columns].isna().sum()


median_wait_weeks    1113
p92_wait_weeks       1113
dtype: int64

The missing values are retained as genuine missing statistics rather than being incorrectly replaced with zero.


## Convert percentage fields into percentage points

In the NHS workbook, percentage fields are stored as decimal proportions:

- `0.7205` represents approximately `72.05%`.

For easier interpretation in SQL outputs and portfolio tables, this project stores them as percentage points on a **0–100 scale**.


In [9]:
percentage_columns = [
    "pct_within_18_weeks",
    "pct_over_52_weeks"
]

for col in percentage_columns:
    df[col] = (
        pd.to_numeric(df[col], errors="coerce") * 100
    ).round(2)

df[percentage_columns].describe()


,pct_within_18_weeks,pct_over_52_weeks
count,3504.000000,3504.000000
mean,50.006498,0.730751
std,34.736506,1.751579
min,0.000000,0.000000
25%,0.000000,0.000000
50%,61.510000,0.000000
75%,76.052500,0.610000
max,100.000000,33.330000


## Create derived KPIs

Two useful measures can be derived directly:

### `over_18_weeks`

Number of incomplete pathways waiting longer than 18 weeks:

`total_waiting - within_18_weeks`

### `pct_over_18_weeks`

Percentage waiting longer than 18 weeks:

`100 - pct_within_18_weeks`


In [10]:
df["over_18_weeks"] = (
    df["total_waiting"] - df["within_18_weeks"]
)

df["pct_over_18_weeks"] = (
    100 - df["pct_within_18_weeks"]
).round(2)

df[
    [
        "total_waiting",
        "within_18_weeks",
        "over_18_weeks",
        "pct_within_18_weeks",
        "pct_over_18_weeks"
    ]
].head()


/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7632/1546886871.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["over_18_weeks"] = (
/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7632/1546886871.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["pct_over_18_weeks"] = (


,total_waiting,within_18_weeks,over_18_weeks,pct_within_18_weeks,pct_over_18_weeks
0,4483,2915,1568,65.02,34.98
1,5901,3467,2434,58.75,41.25
2,8678,5011,3667,57.74,42.26
3,6180,3002,3178,48.58,51.42
4,6249,4311,1938,68.99,31.01


## Add reporting-period metadata

The workbook contains July 2026 data, so add explicit date fields. This becomes essential when we later append multiple monthly extracts for trend analysis.


In [11]:
df["reporting_month"] = pd.Timestamp("2026-07-01")
df["reporting_period"] = "2026-07"

df[["reporting_month", "reporting_period"]].head()


/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7632/2167362762.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["reporting_month"] = pd.Timestamp("2026-07-01")
/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7632/2167362762.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["reporting_period"] = "2026-07"


,reporting_month,reporting_period
0,2026-07-01,2026-07
1,2026-07-01,2026-07
2,2026-07-01,2026-07
3,2026-07-01,2026-07
4,2026-07-01,2026-07


## Label provider-total and treatment-function rows

`C_999` is the provider-level total row. All other rows represent treatment functions.

Creating `row_type` makes it harder to accidentally double-count records later.


In [12]:
df["row_type"] = np.where(
    df["treatment_function_code"].eq("C_999"),
    "provider_total",
    "treatment_function"
)

df["row_type"].value_counts()


/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7632/2093312812.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["row_type"] = np.where(


row_type
treatment_function    3358
provider_total         146
Name: count, dtype: int64

Expected result:

- **146** provider-total rows;
- **3,358** treatment-function rows.

When analysing overall provider waiting lists, filter to `row_type == "provider_total"`.


## Reorder the columns

Place identifiers and reporting information first, the detailed weekly bands next, and summary KPIs last.


In [13]:
weekly_clean = [
    rename_map[col]
    for col in weekly_raw
]

id_columns = [
    "reporting_month",
    "reporting_period",
    "region_code",
    "provider_code",
    "provider_name",
    "treatment_function_code",
    "treatment_function",
    "row_type",
]

kpi_columns = [
    "total_waiting",
    "within_18_weeks",
    "over_18_weeks",
    "pct_within_18_weeks",
    "pct_over_18_weeks",
    "median_wait_weeks",
    "p92_wait_weeks",
    "over_52_weeks",
    "over_65_weeks",
    "over_78_weeks",
    "pct_over_52_weeks",
]

df = df[id_columns + weekly_clean + kpi_columns]

print("Cleaned shape:", df.shape)
df.head()


Cleaned shape: (3504, 124)


,reporting_month,reporting_period,region_code,provider_code,provider_name,treatment_function_code,treatment_function,row_type,wait_0_1_weeks,wait_1_2_weeks,wait_2_3_weeks,wait_3_4_weeks,wait_4_5_weeks,wait_5_6_weeks,wait_6_7_weeks,wait_7_8_weeks,wait_8_9_weeks,wait_9_10_weeks,wait_10_11_weeks,wait_11_12_weeks,wait_12_13_weeks,wait_13_14_weeks,wait_14_15_weeks,wait_15_16_weeks,wait_16_17_weeks,wait_17_18_weeks,wait_18_19_weeks,wait_19_20_weeks,wait_20_21_weeks,wait_21_22_weeks,wait_22_23_weeks,wait_23_24_weeks,wait_24_25_weeks,wait_25_26_weeks,wait_26_27_weeks,wait_27_28_weeks,wait_28_29_weeks,wait_29_30_weeks,wait_30_31_weeks,wait_31_32_weeks,wait_32_33_weeks,wait_33_34_weeks,wait_34_35_weeks,wait_35_36_weeks,wait_36_37_weeks,wait_37_38_weeks,wait_38_39_weeks,wait_39_40_weeks,wait_40_41_weeks,wait_41_42_weeks,wait_42_43_weeks,wait_43_44_weeks,wait_44_45_weeks,wait_45_46_weeks,wait_46_47_weeks,wait_47_48_weeks,wait_48_49_weeks,wait_49_50_weeks,wait_50_51_weeks,wait_51_52_weeks,wait_52_53_weeks,wait_53_54_weeks,wait_54_55_weeks,wait_55_56_weeks,wait_56_57_weeks,wait_57_58_weeks,wait_58_59_weeks,wait_59_60_weeks,wait_60_61_weeks,wait_61_62_weeks,wait_62_63_weeks,wait_63_64_weeks,wait_64_65_weeks,wait_65_66_weeks,wait_66_67_weeks,wait_67_68_weeks,wait_68_69_weeks,wait_69_70_weeks,wait_70_71_weeks,wait_71_72_weeks,wait_72_73_weeks,wait_73_74_weeks,wait_74_75_weeks,wait_75_76_weeks,wait_76_77_weeks,wait_77_78_weeks,wait_78_79_weeks,wait_79_80_weeks,wait_80_81_weeks,wait_81_82_weeks,wait_82_83_weeks,wait_83_84_weeks,wait_84_85_weeks,wait_85_86_weeks,wait_86_87_weeks,wait_87_88_weeks,wait_88_89_weeks,wait_89_90_weeks,wait_90_91_weeks,wait_91_92_weeks,wait_92_93_weeks,wait_93_94_weeks,wait_94_95_weeks,wait_95_96_weeks,wait_96_97_weeks,wait_97_98_weeks,wait_98_99_weeks,wait_99_100_weeks,wait_100_101_weeks,wait_101_102_weeks,wait_102_103_weeks,wait_103_104_weeks,wait_104_plus_weeks,total_waiting,within_18_weeks,over_18_weeks,pct_within_18_weeks,pct_over_18_weeks,median_wait_weeks,p92_wait_weeks,over_52_weeks,over_65_weeks,over_78_weeks,pct_over_52_weeks
0,2026-07-01,2026-07,Y56,R1H,BARTS HEALTH NHS TRUST,C_100,General Surgery Service,treatment_function,257,238,194,211,193,198,193,149,169,105,153,113,131,133,128,122,81,147,105,95,104,71,71,62,75,54,55,50,60,51,26,36,41,40,35,48,32,23,49,45,42,31,46,30,27,31,20,19,15,17,4,11,6,7,2,2,5,4,10,5,1,1,1,1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4483,2915,1568,65.02,34.98,12.522901,39.585778,47,2,0,1.05
1,2026-07-01,2026-07,Y56,R1H,BARTS HEALTH NHS TRUST,C_101,Urology Service,treatment_function,359,239,191,187,207,226,150,296,183,132,220,192,163,185,204,123,113,97,153,121,146,106,126,101,129,108,110,102,114,70,36,69,92,111,130,63,48,52,55,40,26,50,37,38,17,21,22,45,14,23,11,10,4,4,7,6,2,1,1,1,3,2,2,2,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5901,3467,2434,58.75,41.25,14.10049,37.517692,38,3,0,0.64
2,2026-07-01,2026-07,Y56,R1H,BARTS HEALTH NHS TRUST,C_110,Trauma and Orthopaedic Service,treatment_function,396,321,329,298,300,248,272,349,301,210,320,259,225,253,260,213,205,252,204,164,170,160,139,137,159,148,176,175,190,151,52,80,130,146,138,117,101,101,87,89,95,93,76,63,45,51,50,33,24,24,16,9,9,9,8,11,13,4,6,5,3,2,1,0,2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8678,5011,3667,57.74,42.26,14.992308,39.536629,74,1,0,0.85
3,2026-07-01,2026-07,Y56,R1H,BARTS HEALTH NHS TRUST,C_120,Ear Nose and Throat Service,treatment_function,253,229,188,202,232,191,216,182,160,119,145,149,117,165,131,138,66,119,109,90,110,109,113,103,126,109,127,137,125,109,34,58,113,111,99,81,84,84,78,69,90,88,63,66,91,58,57,53,40,56,36,40,32,31,20,23,27,22,19,19,13,11,11,14,4,2,3,4,3,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6180,3002,3178,48.58,51.42,18.807339,46.870175,262,16,0,4.24
4,2026-07-01,2026-07,Y56,R1H,BARTS HEALTH NHS TRUST,C_130,Ophthalmology Service,treatment_function,33

The cleaned table has **3,504 rows × 124 columns**. The increase from 119 columns is due to useful metadata and derived KPIs rather than duplicated raw data.


# Data validation after cleaning

Cleaning is not complete until we confirm that the transformations did not alter the underlying NHS counts incorrectly.


### Unique provider/treatment-function combinations


In [14]:
duplicate_count = df.duplicated(
    subset=["provider_code", "treatment_function_code"]
).sum()

print("Duplicate combinations:", duplicate_count)
assert duplicate_count == 0


Duplicate combinations: 0


### Weekly bands must reconcile to total waiting


In [15]:
calculated_total = df[weekly_clean].sum(axis=1)

total_match = calculated_total.eq(df["total_waiting"])

print("Matching rows:", total_match.sum())
print("Non-matching rows:", (~total_match).sum())

assert total_match.all()


Matching rows: 3504
Non-matching rows: 0


### First 18 weekly bands must reconcile to within-18-week total


In [16]:
within_18_columns = [
    f"wait_{week}_{week + 1}_weeks"
    for week in range(18)
]

calculated_within_18 = df[within_18_columns].sum(axis=1)

within_18_match = calculated_within_18.eq(
    df["within_18_weeks"]
)

print("Matching rows:", within_18_match.sum())
print("Non-matching rows:", (~within_18_match).sum())

assert within_18_match.all()


Matching rows: 3504
Non-matching rows: 0


### Validate 52+, 65+ and 78+ published totals

The detailed bands allow these long-wait totals to be independently reconstructed.


In [17]:
long_wait_checks = {
    "over_52_weeks": weekly_clean[52:],
    "over_65_weeks": weekly_clean[65:],
    "over_78_weeks": weekly_clean[78:],
}

for published_column, band_columns in long_wait_checks.items():
    calculated = df[band_columns].sum(axis=1)
    matches = calculated.eq(df[published_column])

    print(
        f"{published_column}: "
        f"{matches.sum()} matching / {len(df)} rows"
    )

    assert matches.all()


over_52_weeks: 3504 matching / 3504 rows
over_65_weeks: 3504 matching / 3504 rows
over_78_weeks: 3504 matching / 3504 rows


All published long-wait totals reconcile against the detailed weekly bands for all **3,504 records**.


### Logical checks

Counts should not be negative, `within_18_weeks` should not exceed `total_waiting`, and percentages should stay between 0 and 100.


In [18]:
count_columns = (
    weekly_clean
    + [
        "total_waiting",
        "within_18_weeks",
        "over_18_weeks",
        "over_52_weeks",
        "over_65_weeks",
        "over_78_weeks"
    ]
)

assert not (df[count_columns] < 0).any().any()
assert (df["within_18_weeks"] <= df["total_waiting"]).all()

for col in [
    "pct_within_18_weeks",
    "pct_over_18_weeks",
    "pct_over_52_weeks"
]:
    assert df[col].between(0, 100).all()

print("All logical validation checks passed.")


All logical validation checks passed.


## Review the cleaned dataset


In [19]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 3504 entries, 0 to 3503
Columns: 124 entries, reporting_month to pct_over_52_weeks
dtypes: Float64(2), datetime64[us](1), float64(3), int64(111), str(7)
memory usage: 3.3 MB


In [20]:
df[
    [
        "reporting_period",
        "provider_code",
        "provider_name",
        "treatment_function_code",
        "treatment_function",
        "row_type",
        "total_waiting",
        "within_18_weeks",
        "over_18_weeks",
        "pct_within_18_weeks",
        "median_wait_weeks",
        "p92_wait_weeks",
        "over_52_weeks",
        "over_65_weeks",
        "over_78_weeks"
    ]
].head(10)


,reporting_period,provider_code,provider_name,treatment_function_code,treatment_function,row_type,total_waiting,within_18_weeks,over_18_weeks,pct_within_18_weeks,median_wait_weeks,p92_wait_weeks,over_52_weeks,over_65_weeks,over_78_weeks
0,2026-07,R1H,BARTS HEALTH NHS TRUST,C_100,General Surgery Service,treatment_function,4483,2915,1568,65.02,12.522901,39.585778,47,2,0
1,2026-07,R1H,BARTS HEALTH NHS TRUST,C_101,Urology Service,treatment_function,5901,3467,2434,58.75,14.10049,37.517692,38,3,0
2,2026-07,R1H,BARTS HEALTH NHS TRUST,C_110,Trauma and Orthopaedic Service,treatment_function,8678,5011,3667,57.74,14.992308,39.536629,74,1,0
3,2026-07,R1H,BARTS HEALTH NHS TRUST,C_120,Ear Nose and Throat Service,treatment_function,6180,3002,3178,48.58,18.807339,46.870175,262,16,0
4,2026-07,R1H,BARTS HEALTH NHS TRUST,C_130,Ophthalmology Service,treatment_function,6249,4311,1938,68.99,11.591286,33.791613,4,0,0
5,2026-07,R1H,BARTS HEALTH NHS TRUST,C_140,Oral Surgery Service,treatment_function,3490,2092,1398,59.94,13.155963,41.103704,73,6,0
6,2026-07,R1H,BARTS HEALTH NHS TRUST,C_150,Neurosurgical Service,treatment_function,1467,757,710,51.60,17.539216,43.868889,26,0,0
7,2026-07,R1H,BARTS HEALTH NHS TRUST,C_160,Plastic Surgery Service,treatment_function,1394,774,620,55.52,14.612903,47.206667,60,5,0
8,2026-07,R1H,BARTS HEALTH NHS TRUST,C_170,Cardiothoracic Surgery Service,treatment_function,4,4,0,100.00,<NA>,<NA>,0,0,0
9,2026-07,R1H,BARTS HEALTH NHS TRUST,C_300,General Internal Medicine Service,treatment_function,63,38,25,60.32,14.125,31.98,0,0,0


## Check Gloucestershire Hospitals after cleaning

This verifies that our local case-study provider survived the transformations correctly.


In [21]:
gloucestershire = df[
    (df["provider_code"] == "RTE") &
    (df["row_type"] == "provider_total")
][
    [
        "reporting_period",
        "provider_name",
        "total_waiting",
        "within_18_weeks",
        "over_18_weeks",
        "pct_within_18_weeks",
        "pct_over_18_weeks",
        "median_wait_weeks",
        "p92_wait_weeks",
        "over_52_weeks",
        "over_65_weeks",
        "over_78_weeks",
        "pct_over_52_weeks"
    ]
]

gloucestershire


,reporting_period,provider_name,total_waiting,within_18_weeks,over_18_weeks,pct_within_18_weeks,pct_over_18_weeks,median_wait_weeks,p92_wait_weeks,over_52_weeks,over_65_weeks,over_78_weeks,pct_over_52_weeks
791,2026-07,GLOUCESTERSHIRE HOSPITALS NHS FOUNDATION TRUST,68792,49568,19224,72.05,27.95,10.787426,33.831254,24,0,0,0.03


## Export the processed dataset

The cleaned CSV becomes the source for the next stages of the project. It can be loaded into SQL and Power BI without repeating the raw Excel cleaning steps.


In [22]:
df.to_csv(
    output_file,
    index=False
)

print(f"Saved cleaned dataset to: {output_file}")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")


Saved cleaned dataset to: ../data/processed/rtt_incomplete_provider_jul_2026_clean.csv
Rows: 3,504
Columns: 124
